### Testing the code over `sample_candidates.json`

In [2]:
import json
import pandas as pd

# Loading the sample data
with open('../data/raw/sample_candidates.json', 'r', encoding='utf-8') as f:
    sample_data = json.load(f)

# 2. candidate's key
first_candidate = sample_data[0]
profile = first_candidate.get('profile', {})
print("=== Candidate Root Keys ===")
print(first_candidate.keys())

# 3. Summary
print("=== Candidate Quick Profile ===")
print(f"ID: {first_candidate.get('candidate_id')}")
print(f"Experience Years: {profile.get('years_of_experience')}")
print(f"Current Location: {profile.get('location')}")
print(f"Willing to Relocate: {first_candidate.get('willing_to_relocate')}")

# 4. Checking what the redrob_signals object looks like inside
print("\n=== redrob_signals ===")
signals = first_candidate.get('redrob_signals', {})
# first 5 signals to see
print(dict(list(signals.items())[:5]))

=== Candidate Root Keys ===
dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])
=== Candidate Quick Profile ===
ID: CAND_0000001
Experience Years: 6.9
Current Location: Toronto
Willing to Relocate: None

=== redrob_signals ===
{'profile_completeness_score': 86.9, 'signup_date': '2025-10-16', 'last_active_date': '2026-05-20', 'open_to_work_flag': True, 'profile_views_received_30d': 23}


### Filtering the candidtes over experience and relocating

In [3]:
# Converting the whole sample list to a pandas DataFrame
df = pd.DataFrame(sample_data)


print(f"Total initial sample candidates: {len(df)}")

# FILTER 1: Experience Constraint (5 to 9 years)
exp_condition = (df['profile'].apply(lambda x: x.get('years_of_experience', 0)) >= 5) & (df['profile'].apply(lambda x: x.get('years_of_experience', 0)) <= 9)
df_filtered = df[exp_condition]
print(f"Candidates remaining after Experience Filter (5-9 years): {len(df_filtered)}")

# FILTER 2: Location (Noida/Pune or willing to relocate) 
target_cities = ['noida', 'pune']

def matches_location_criteria(row):
    loc = str(row['location']).strip().lower()
    willing_to_relocate = row['willing_to_relocate']
    
    # Condition A: They already live in Noida or Pune
    if loc in target_cities:
        return True
    
    # Condition B: They are willing to relocate
    if willing_to_relocate is True or willing_to_relocate == 1:
        return True
        
    return False

# Apply our location function row by row
def matches_location_criteria(row):
    profile = row.get('profile', {}) or {}
    loc = str(profile.get('location', '')).strip().lower()
    willing_to_relocate = row.get('redrob_signals', {}).get('willing_to_relocate', False)

    if loc in target_cities:
        return True

    if willing_to_relocate is True or willing_to_relocate == 1:
        return True

    return False

location_mask = df_filtered.apply(matches_location_criteria, axis=1)
df_final_filtered = df_filtered[location_mask]

print(f"Candidates remaining after Location Filter: {len(df_final_filtered)}")

Total initial sample candidates: 50
Candidates remaining after Experience Filter (5-9 years): 21
Candidates remaining after Location Filter: 9


### Filtering the candidates over redrob_signals

In [5]:
def calculate_behavioral_score(row):
    # Safely get the nested redrob_signals dictionary
    signals = row.get('redrob_signals', {})
    if not isinstance(signals, dict) or not signals:
        return 0.5  # Neutral default if empty
        
    score = 1.0  # Start with a baseline perfect score
    
    # 1. Recruiter Response Rate (From JSON: 'recruiter_response_rate')
    response_rate = signals.get('recruiter_response_rate', 1.0)
    if response_rate < 0.30:
        score -= 0.3
        
    # 2. Activity / Age (From JSON: 'profile_completeness_score')
    # Note: The JSON key is 'profile_completeness_score', not 'profile_completeness'
    completeness = signals.get('profile_completeness_score', 100.0)
    if completeness < 50.0:
        score -= 0.2
        
    # 3. Last Active Date (Checking if they are completely inactive or cold)
    # If notice period is exceptionally long, give a slight down-weight
    notice_period = signals.get('notice_period_days', 0)
    if notice_period > 60:
        score -= 0.1

    return max(0.0, min(1.0, score))

# Apply the function to the surviving rows
df_final_filtered = df_final_filtered.copy()
df_final_filtered['behavioral_score'] = df_final_filtered.apply(calculate_behavioral_score, axis=1)

print("Behavioral scores computed safely without dictionary errors!")

Behavioral scores computed safely without dictionary errors!


### Chedking for Honeypots (Fake Profiles)

In [6]:
def evaluate_honeypot_rules(row):
    # Safely unpack the profile dictionary
    profile = row.get('profile', {})
    title = str(profile.get('current_title', '')).lower()
    
    # Honeypot Rule A: Non-technical titles listing an impossible amount of technical skills
    non_tech_keywords = ['marketing', 'sales', 'hr', 'recruiter', 'content writer', 'social media']
    is_non_tech_job = any(kw in title for kw in non_tech_keywords)
    
    skills_list = row.get('skills', [])
    
    # Trap: A Marketing Manager profile claiming more than 15 hyper-technical AI skills
    if is_non_tech_job and len(skills_list) > 15:
        return True # Flagged as a fake honeypot
        
    # Honeypot Rule B: Experience anomaly checks
    # If years of experience is 0 or negative but they list advanced skills, it's false
    yoe = profile.get('years_of_experience', 0)
    if yoe <= 0 and len(skills_list) > 5:
        return True
        
    return False

df_final_filtered['is_honeypot'] = df_final_filtered.apply(evaluate_honeypot_rules, axis=1)

# Drop anyone flagged as a honeypot
df_clean = df_final_filtered[df_final_filtered['is_honeypot'] == False].copy()

print(f"Honeypot check complete! Candidates remaining: {len(df_clean)}")

Honeypot check complete! Candidates remaining: 9


### Offline test similarity matching

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Define the core Job Description criteria text
job_description_text = """
Senior AI Engineer Founding Team Series A AI-native talent intelligence platform.
Deep technical depth in modern ML systems embeddings retrieval ranking LLMs fine-tuning.
Backend data hybrid Python SQL Spark Airflow software data pipelines infrastructure.
"""

def extract_full_text(row):
    profile = row.get('profile', {})
    headline = str(profile.get('headline', ''))
    summary = str(profile.get('summary', ''))
    
    # Combine candidate background text into a single string matrix
    return f"{headline} {summary}"

# Extract candidate background text
df_clean['combined_text'] = df_clean.apply(extract_full_text, axis=1)

# 2. Compute Cosine Similarity completely offline
vectorizer = TfidfVectorizer(stop_words='english')

# Process similarity row by row to remain fast and safe
def compute_similarity(text):
    if not text.strip():
        return 0.0
    try:
        tfidf_matrix = vectorizer.fit_transform([job_description_text, text])
        return cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    except:
        return 0.0

df_clean['text_match_score'] = df_clean['combined_text'].apply(compute_similarity)

print("Offline Text Similarity Matrix generated via Scikit-Learn!")

Offline Text Similarity Matrix generated via Scikit-Learn!


### Combining everything (Calculating Score)

In [9]:
# Compute Final Weighted Equation
# 60% Weight to technical text match, 40% Weight to candidate active behavior
df_clean['final_score'] = (df_clean['text_match_score'] * 0.6) + (df_clean['behavioral_score'] * 0.4)

# Sort candidates deterministically from highest score to lowest
df_ranked_output = df_clean.sort_values(by='final_score', ascending=False)

print("Top ranked sample candidates:")
print(df_ranked_output[['candidate_id', 'text_match_score', 'behavioral_score', 'final_score']])

Top ranked sample candidates:
    candidate_id  text_match_score  behavioral_score  final_score
30  CAND_0000031          0.170452               1.0     0.502271
45  CAND_0000046          0.055985               1.0     0.433591
6   CAND_0000007          0.054580               1.0     0.432748
41  CAND_0000042          0.045482               1.0     0.427289
23  CAND_0000024          0.043546               1.0     0.426128
31  CAND_0000032          0.137639               0.7     0.362583
42  CAND_0000043          0.126279               0.6     0.315768
37  CAND_0000038          0.146365               0.4     0.247819
5   CAND_0000006          0.044290               0.4     0.186574
